# 03 — Evaluation (local, CPU)

Computes every metric from per-run `preds.csv`. Per run, the temperature T and the 95%-sensitivity threshold
are fitted on **val** and frozen; test is only scored. Nothing in this notebook fits anything on test.

Inputs (repo root):
* `02b_repredict_output/` — FP32 re-predictions of the 45 main runs (preferred). Falls back to the fp16
  predictions in `02_train_<arch>_output/` with a warning.
* `02_train_extra_splits_output/` — optional: 2 extra realisations of image_random / grouped (5 archs, seed 0).

All metrics use the `logit` column (float32 probabilities saturate at 1.0 and create ties).
Outputs go to `results/` (small CSVs).

In [ ]:
from glob import glob
from pathlib import Path
import numpy as np, pandas as pd
from ppneu import metrics as M, calibrate as C, ensemble as E

pd.set_option("display.width", 250)
REPO = Path("..") if Path("../manifests").exists() else Path(".")
OUT = REPO / "results"; OUT.mkdir(exist_ok=True)
N_BOOT = 2000
MAIN = ["official", "image_random", "grouped"]

FP32 = REPO / "02b_repredict_output"
TRAIN = sorted(glob(str(REPO / "02_train_*_output")))
EXTRA = [r for r in TRAIN if "extra_splits" in r]
if FP32.exists():
    MAIN_ROOTS = [str(FP32)]
else:
    MAIN_ROOTS = [r for r in TRAIN if r not in EXTRA]
    print("WARNING: FP32 re-predictions not found, using fp16 training predictions")

preds = M.load_preds(MAIN_ROOTS + EXTRA)
manifests = {s: pd.read_csv(REPO / "manifests" / f"{s}.csv") for s in sorted(preds.split.unique())}
n_runs = preds.groupby(["split", "arch", "seed"]).ngroups
print("roots:", [Path(r).name for r in MAIN_ROOTS + EXTRA], "|", n_runs, "runs |", list(manifests))

## Integrity: predictions match the frozen manifests; precision of the logits

In [ ]:
for (s, a, k), run in preds.groupby(["split", "arch", "seed"]):
    m = manifests[s].set_index("image_id")
    for fold in ("val", "test"):
        r = run[run.fold == fold]
        assert set(r.image_id) == set(m.index[m.fold == fold]), (s, a, k, fold)
        assert (r.y.values == m.loc[r.image_id, "y"].values).all(), (s, a, k, fold)
ties = preds[preds.fold == "test"].groupby(["split", "arch", "seed"]).logit.apply(lambda z: 1 - z.nunique() / len(z))
print("ok | share of tied test logits per run: median %.4f, max %.4f (fp16 gave ~0.08)" % (ties.median(), ties.max()))

## Per-run metrics (T and threshold fitted on val)

In [ ]:
runs = M.evaluate_all(preds, target_sens=0.95)
runs.to_csv(OUT / "run_metrics.csv", index=False)
main = runs[runs.split.isin(MAIN)]
summary = M.summarize(main)
summary.round(4).to_csv(OUT / "summary_by_split_arch.csv")
display(summary.round(4))

## Choice of operating point, decided on val only
Each run's val fold is split in random halves 20 times: the threshold is fitted on one half and scored on the other.
Shown: spread (SD) across the 15 runs of each split. The 95%-sensitivity rule is the most stable, so it is the primary
operating point; 0.5 is reported as secondary.

In [ ]:
rows = []
for (s, a, k), run in preds[(preds.fold == "val") & preds.split.isin(MAIN)].groupby(["split", "arch", "seed"]):
    cf = C.crossfit_threshold_rules(run.y, run.logit, target=0.95)
    rows.append(cf.groupby("rule")[["sens", "spec"]].mean().assign(split=s, arch=a, seed=k).reset_index())
cf = pd.concat(rows, ignore_index=True)
rule_sd = cf.groupby(["split", "rule"])[["sens", "spec"]].std().unstack("rule")
rule_sd.round(4).to_csv(OUT / "threshold_rule_val_crossfit.csv")
display(rule_sd.round(4))

## Table 2a — leakage: image_random vs grouped (different test sets)
Mean over seeds; `all_seeds_higher` = every image_random seed beats every grouped seed.

In [ ]:
a = main.pivot_table(index="arch", columns="split", values="auroc")
lo = main.groupby(["arch", "split"]).auroc.min().unstack()
hi = main.groupby(["arch", "split"]).auroc.max().unstack()
t2a = pd.DataFrame({
    "auroc_image_random": a.image_random, "auroc_grouped": a.grouped,
    "error_ratio": (1 - a.grouped) / (1 - a.image_random),
    "all_seeds_higher": lo.image_random > hi.grouped,
    "sens_image_random": main[main.split == "image_random"].groupby("arch").sens.mean(),
    "sens_grouped": main[main.split == "grouped"].groupby("arch").sens.mean(),
})
t2a.round(4).to_csv(OUT / "table2a_leakage_splits.csv")
display(t2a.round(4))

## Table 2a robustness — three split realisations
Training seed 0 only, so every realisation is compared the same way (s0 = the frozen manifests).

In [ ]:
if EXTRA:
    r0 = runs[runs.seed == 0].copy()
    r0["condition"] = r0.split.str.replace(r"_s\d$", "", regex=True)
    r0["realisation"] = "s" + r0.split.str.extract(r"_s(\d)$")[0].fillna("0")
    r0 = r0[r0.condition.isin(["image_random", "grouped"])]
    w = r0.pivot_table(index=["arch", "realisation"], columns="condition", values="auroc")
    w["diff"] = w.image_random - w.grouped
    w["error_ratio"] = (1 - w.grouped) / (1 - w.image_random)
    w.round(4).to_csv(OUT / "table2a_split_realisations.csv")
    display(w.round(4))
    print(f"image_random > grouped in {(w['diff'] > 0).sum()} of {len(w)} (arch, realisation) pairs")
    display(w.groupby("realisation")[["diff", "error_ratio"]].mean().round(4))
else:
    print("no extra split realisations yet")

## Table 2b — leakage within one test set (image_random)
Test images whose patient cluster is also in train ("seen") vs not. Prevalence differs a lot between the
two groups, so errors are compared **per class**. Each image's error rate is averaged over the 15 runs
(5 archs × 3 seeds), then resampled two ways: by image (`_img`) and by patient cluster (`_clu`).
Images of one child are correlated, so the cluster interval is the one to report.

In [ ]:
seen = M.seen_in_train(manifests["image_random"])
cluster = manifests["image_random"].set_index("image_id").split_group
thr = runs.set_index(["split", "arch", "seed"]).threshold_logit
t = preds[(preds.split == "image_random") & (preds.fold == "test")].copy()
t["thr"] = [thr[("image_random", a, k)] for a, k in zip(t.arch, t.seed)]
t["err"] = ((t.logit >= t.thr) != (t.y == 1)).astype(float)
t["err_05"] = ((t.logit >= 0) != (t.y == 1)).astype(float)
t["seen"] = seen.reindex(t.image_id).to_numpy(dtype=bool)
img = t.groupby("image_id").agg(y=("y", "first"), seen=("seen", "first"),
                                err=("err", "mean"), err_05=("err_05", "mean"))
img["cluster"] = cluster.reindex(img.index).to_numpy()

mean_err = lambda y, e: float(np.mean(e))
rows = []
for cls, name in [(1, "pneumonia"), (0, "normal")]:
    for col, rule in [("err", "val 95% sens"), ("err_05", "0.5")]:
        s_, u_ = img[(img.y == cls) & img.seen], img[(img.y == cls) & ~img.seen]
        row = {"class": name, "rule": rule, "n_unseen": len(u_), "n_seen": len(s_),
               "clusters_unseen": u_.cluster.nunique(), "clusters_seen": s_.cluster.nunique(),
               "err_unseen_%": 100 * u_[col].mean(), "err_seen_%": 100 * s_[col].mean()}
        for tag, ga, gb in [("img", None, None), ("clu", u_.cluster, s_.cluster)]:
            d, l, h, p = M.bootstrap_diff(mean_err, u_.y, u_[col], s_.y, s_[col],
                                          groups_a=ga, groups_b=gb, n_boot=N_BOOT)
            row.update({"diff_pts": 100 * d, f"ci_{tag}": f"{100 * l:.2f} to {100 * h:.2f}", f"p_{tag}": p})
        rows.append(row)
t2b = pd.DataFrame(rows)
t2b.round(3).to_csv(OUT / "table2b_leakage_seen_unseen.csv", index=False)
display(t2b.round(3))

per_arch = (t.assign(cls=t.y.map({1: "pneumonia", 0: "normal"}))
              .groupby(["arch", "cls", "seen"]).err.mean().mul(100).unstack(["cls", "seen"]).round(2))
display(per_arch)

## Calibration
T > 1 means overconfident. Temperature scaling changes probabilities only, so it is judged by log-loss and Brier;
binned ECE barely moves because most predictions sit near 0 or 1.

In [ ]:
cal = main.groupby("split")[["T", "nll", "nll_ts", "brier", "brier_ts", "ece", "ece_ts"]].agg(["mean", "std"])
cal.round(4).to_csv(OUT / "calibration_by_split.csv")
display(cal.round(4))

## Official test set (distribution shift)
Same models, two operating points. The val-fitted thresholds come from a clean, grouped val fold.

In [ ]:
off = main[main.split == "official"].groupby("arch")[
    ["auroc", "sens", "spec", "sens_05", "spec_05", "acc_05", "nll", "ece", "threshold_prob"]].agg(["mean", "std"])
off.round(4).to_csv(OUT / "official_shift.csv")
display(off.round(4))

## Per-run AUROC with bootstrap 95% CI (seed 0)

In [ ]:
rows = []
sel = preds[(preds.fold == "test") & (preds.seed == 0) & preds.split.isin(MAIN)]
for (s, a, k), run in sel.groupby(["split", "arch", "seed"]):
    est, lo, hi = M.bootstrap_ci(M.auroc, run.y, run.logit, n_boot=N_BOOT)
    rows.append({"split": s, "arch": a, "auroc": est, "ci_lo": lo, "ci_hi": hi})
ci = pd.DataFrame(rows)
ci.round(4).to_csv(OUT / "auroc_ci_seed0.csv", index=False)
display(ci.pivot_table(index="arch", columns="split", values=["auroc", "ci_lo", "ci_hi"]).round(4))

## Thesis ensemble, rebuilt without the test set
ResNet-50 + DenseNet-121 + Inception-v3. Each member is temperature-scaled on val, then soft-voted with
`ens_equal` (1/3 each, primary) or `ens_val` (weights minimising val log-loss). The ensemble is then evaluated
exactly like a single model. The thesis reported 94.67% accuracy on the official test set with weights and
model choice tuned on that test set; `acc_05` on `official` is the honest counterpart.

In [ ]:
ens, fitted = [], []
for s in MAIN:
    for wt in ("equal", "val"):
        e, f = E.build(preds, s, weighting=wt)
        ens.append(e); fitted.append(f)
ens, fitted = pd.concat(ens, ignore_index=True), pd.concat(fitted, ignore_index=True)
ens_runs = M.evaluate_all(ens)
fitted.round(3).to_csv(OUT / "ensemble_fitted_on_val.csv", index=False)

cols = ["auroc", "sens", "spec", "acc_05", "nll", "brier"]
cmp = pd.concat([main[main.arch.isin(E.THESIS_ARCHS)], ens_runs]).groupby(["split", "arch"])[cols].agg(["mean", "std"])
cmp.round(4).to_csv(OUT / "thesis_ensemble.csv")
display(cmp.round(4))
display(fitted.round(3))